<a href="https://colab.research.google.com/github/SohamManik/llm-huggingface/blob/main/LLM11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
from datasets import load_dataset


raw_dataset = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/eriktks/conll2003@refs/convert/parquet/conll2003/train/*.parquet",
        "validation": "hf://datasets/eriktks/conll2003@refs/convert/parquet/conll2003/validation/*.parquet",
        "test": "hf://datasets/eriktks/conll2003@refs/convert/parquet/conll2003/test/*.parquet",
    }
)


In [57]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [58]:
ner_features = raw_dataset["train"].features["ner_tags"]
ner_features

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

In [59]:
label_names = ner_features.feature.names

label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [60]:



words = raw_dataset["train"][0]["tokens"]
labels = raw_dataset["train"][0]["ner_tags"]

line1 = " "
line2 = " "

for word , label in zip(words , labels):
    full_label = label_names[label]
    max_length = max(len(word), len(full_label))
    line1 += word + " "*(max_length - len(word) + 1)
    line2 += full_label + " "*(max_length - len(full_label) + 1)

print(line1)
print(line2)


 EU    rejects German call to boycott British lamb . 
 B-ORG O       B-MISC O    O  O       B-MISC  O    O 


In [61]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [62]:
inputs = tokenizer(raw_dataset["train"][0]["tokens"] , is_split_into_words = True)

In [63]:
inputs.tokens()

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [64]:
def align_labels(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:

            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)

        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels




In [65]:
labels = raw_dataset["train"][0]["ner_tags"]

word_ids = inputs.word_ids()

print(labels)
print(align_labels(labels , word_ids))

[3, 0, 7, 0, 0, 0, 7, 0, 0]
[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]


In [66]:
def tokenize_and_align(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'] , truncation = True , is_split_into_words = True
    )
    all_labels = examples["ner_tags"]
    new_labels = []

    for i , labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)

        new_labels.append(align_labels(labels , word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs



In [67]:
tokenized_datasets = raw_dataset.map(
    tokenize_and_align,
    batched = True,
    remove_columns = raw_dataset["train"].column_names,
)

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [68]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer = tokenizer)

In [69]:
%pip install seqeval

In [70]:
!pip install evaluate
import evaluate

metric = evaluate.load("seqeval")

In [71]:
import numpy as np

def compute_metrics(eval_preds):
    logits , labels = eval_preds
    predictions = np.argmax(logits , axis = -1 )

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metrics.compute(predictions = true_predictions , references = true_labels)
    return {
        'precision' : all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1_score": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],


    }

In [72]:
id2label = {i: label for i , label in enumerate(label_names)}
label2id = {v: k for k , v in id2label.items()}

In [73]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label = id2label,
    label2id = label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [74]:
model.config.num_labels

9

In [75]:
from huggingface_hub import notebook_login
notebook_login()


In [76]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,
    num_train_epochs = 3,
    weight_decay = 0.01,
    push_to_hub = True,
)

In [77]:
from transformers import Trainer
import numpy as np


def compute_metrics(eval_preds):
    logits , labels = eval_preds
    predictions = np.argmax(logits , axis = -1 )

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Corrected from 'metrics.compute' to 'metric.compute'
    all_metrics = metric.compute(predictions = true_predictions , references = true_labels)
    return {
        'precision' : all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1_score": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

# Ensure the Hugging Face token is explicitly set for TrainingArguments
if args.push_to_hub and args.hub_token is None:
    hf_token = get_token()
    if hf_token:
        args.hub_token = hf_token
    else:
        # If no token is found, you might want to print a warning or handle it differently
        print("Warning: Hugging Face token not found. Push to hub might fail.")

trainer = Trainer(
    model = model,
    args = args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    processing_class= tokenizer,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1 Score,Accuracy
1,0.075197,0.061597,0.908972,0.936048,0.922312,0.982987
2,0.034867,0.067779,0.928690,0.944631,0.936593,0.985386
3,0.021223,0.061972,0.928713,0.949344,0.938915,0.986254


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5268, training_loss=0.06638026282745686, metrics={'train_runtime': 547.6172, 'train_samples_per_second': 76.921, 'train_steps_per_second': 9.62, 'total_flos': 920771584279074.0, 'train_loss': 0.06638026282745686, 'epoch': 3.0})

In [78]:
trainer.push_to_hub(commit_message = "Training_complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ned-ner/model.safetensors:  48%|####8     |  208MB /  431MB            

  ...ned-ner/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/sohammanik/bert-finetuned-ner/commit/0498a2ce183fd3bac984cb7a33a5069995540a71', commit_message='Training_complete', commit_description='', oid='0498a2ce183fd3bac984cb7a33a5069995540a71', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sohammanik/bert-finetuned-ner', endpoint='https://huggingface.co', repo_type='model', repo_id='sohammanik/bert-finetuned-ner'), pr_revision=None, pr_num=None)

In [79]:
from transformers import pipeline

checkpoint = "sohammanik/bert-finetuned-ner"

token_classifier = pipeline(
    "token-classification" , model = checkpoint , aggregation_strategy = "simple"

)

token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")



config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9979982),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9745865),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9971129),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]